# Optuna: Random Forest
**tabular › tuning_classification** · notebook `random_forest_optuna`

**Kapan dipakai:** Bila Random Forest sudah jadi model dasar dan ingin mencoba memerasnya sebelum pindah ke boosting.

**Jebakan umum:** Gain tuning RF biasanya kecil dibanding pindah ke boosting. max_features paling berpengaruh.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/classification/titanic_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/classification/titanic_test.csv"      # None kalau tidak ada file test
TARGET     = "survived"
ID_COL     = 'id'
DROP_COLS  = ['alive']
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "f1_macro"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "random_forest_optuna"
TASK       = "classification"
KIND       = "tree"
OUT = out_dir("tabular", "tuning_classification", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df[TARGET])
print("kelas:", dict(enumerate(le.classes_)))
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Ruang pencarian
Fungsi `make_model(trial)` mendefinisikan hiperparameter yang dicari dan rentangnya.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def make_model(trial):
    return RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 100, 500),
        max_depth=trial.suggest_int("max_depth", 3, 25),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20),
        max_features=trial.suggest_float("max_features", 0.2, 1.0),
        n_jobs=-1, random_state=42)

## 5. Jalankan pencarian (Optuna, TPE)
Objective = rata-rata skor CV pada data **training**. Naikkan `N_TRIALS`/`TIMEOUT` saat lomba.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TRIALS, TIMEOUT = 15, 180          # demo cepat; saat lomba mis. 100 trial / 900 detik

def objective(trial):
    pipe = build_pipeline(make_model(trial), NUM, CAT, kind=KIND)
    _, s = cv_score(pipe, Xtr, ytr, TASK, SCORING, N_SPLITS, SEED, groups=GTR)
    return s["valid_mean"]            # sklearn scoring: lebih besar = lebih baik, jadi maximize

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT)
print("trial selesai:", len(study.trials))
print("skor CV terbaik:", round(study.best_value, 4))
print("param terbaik:", study.best_params)

## 6. Lihat jalannya pencarian
Grafik kiri: apakah masih membaik atau sudah datar. Grafik kanan: hiperparameter mana yang paling berpengaruh.

In [ ]:
vals = pd.Series([t.value for t in study.trials])
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(vals, ".", alpha=.5, label="trial"); ax[0].plot(vals.cummax(), "r-", label="terbaik sejauh ini")
ax[0].set(xlabel="trial", ylabel=SCORING, title="Riwayat tuning"); ax[0].legend()
try:
    imp = optuna.importance.get_param_importances(study)
    pd.Series(imp).sort_values().plot.barh(ax=ax[1], title="Pengaruh hiperparameter")
except Exception as ex:
    ax[1].set_title(f"importance tidak tersedia: {type(ex).__name__}")
fig.tight_layout(); fig.savefig(OUT / "tuning.png", dpi=130); plt.show()

## 7. Uji hasil pada validation
Selisih besar antara skor CV terbaik dan validation = tanda overfit ke fold. Validation ini tidak dipakai saat mencari.

In [ ]:
pipe = build_pipeline(make_model(optuna.trial.FixedTrial(study.best_params)), NUM, CAT, kind=KIND).fit(Xtr, ytr)
print(f"CV terbaik (train): {study.best_value:.4f}")

In [ ]:
pred = pipe.predict(Xva)
proba = pipe.predict_proba(Xva)
metrics = classification_metrics(yva, pred, proba, classes=np.arange(proba.shape[1]))
display(show_table(metrics))

In [ ]:
save_json({"best_cv": study.best_value, "best_params": study.best_params, "valid": metrics, "n_trials": len(study.trials)}, OUT / "metrics.json")

## Latih ulang pada semua data → submission → simpan

In [ ]:
final = build_pipeline(make_model(optuna.trial.FixedTrial(study.best_params)), NUM, CAT, kind=KIND).fit(X, y)
if test is not None:
    Xte, _, _ = prepare_features(test, cols)
    Xte = Xte.reindex(columns=X.columns)
    ids = test[ID_COL] if ID_COL else np.arange(len(test))
    sub = make_submission(ids, le.inverse_transform(final.predict(Xte)), ID_COL or "id", TARGET, OUT / "submission.csv")
    display(sub.head(3))
save_model({"pipe": final, "cols": cols}, OUT / "model.joblib")
reload_pipe = load_model(OUT / "model.joblib")["pipe"]
assert np.allclose(reload_pipe.predict(X.head(50)), final.predict(X.head(50))), "model muat-ulang beda"
print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan dengan model tanpa tuning di `notebooks/tabular/<task>/`. Kalau naiknya kecil, waktu lebih baik dipakai untuk **feature engineering** atau **ensemble**.
- Tuning tidak memperbaiki kebocoran data atau fitur yang buruk.